# Step 8: Analisis Hasil

Analisis mendalam untuk mendukung penulisan **Bab 4** skripsi:
- Perbandingan hasil dengan paper C23
- Analisis pola kesalahan prediksi
- Hubungan fitur penting dengan domain bridge
- Ekspor tabel dan grafik untuk skripsi

**Prasyarat:** Jalankan semua step sebelumnya (01–07).

In [ ]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

DATA_PROCESSED = Path(ROOT) / "data" / "processed"
RESULTS        = Path(ROOT) / "results"
FIGURES_DIR    = RESULTS / "figures"
METRICS_DIR    = RESULTS / "metrics"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root proyek : {ROOT}")
print("Setup selesai.")

In [ ]:
from model import load_model
from features import FEATURE_COLS
from model import prepare_features
from sklearn.model_selection import train_test_split

MODEL_PATH = METRICS_DIR / "rf_model.pkl"
SPLIT_PATH = METRICS_DIR / "dataset_split.pkl"

model = load_model(MODEL_PATH)
print(f"Model dimuat dari: {MODEL_PATH}")

if SPLIT_PATH.exists():
    split_data   = joblib.load(SPLIT_PATH)
    X_train      = split_data["X_train"]
    X_test       = split_data["X_test"]
    y_suit_train = split_data["y_suit_train"]
    y_suit_test  = split_data["y_suit_test"]
    y_cat_train  = split_data["y_cat_train"]
    y_cat_test   = split_data["y_cat_test"]
else:
    DATASET_CSV = DATA_PROCESSED / "bridge_dataset.csv"
    df = pd.read_csv(DATASET_CSV).dropna(subset=["best_contract_strain", "best_contract_category"])
    available_feat = [c for c in FEATURE_COLS if c in df.columns]
    X      = prepare_features(df, available_feat)
    y_suit = df["best_contract_strain"]
    y_cat  = df["best_contract_category"]
    X_train, X_test, y_suit_train, y_suit_test = train_test_split(
        X, y_suit, test_size=0.2, stratify=y_suit, random_state=42
    )
    _, _, y_cat_train, y_cat_test = train_test_split(
        X, y_cat, test_size=0.2, stratify=y_suit, random_state=42
    )

print(f"Test set: {X_test.shape[0]} sampel, {X_test.shape[1]} fitur")

## 8.1 Tabel Perbandingan dengan C23 Paper

Untuk **Tabel Bab 4** skripsi.

In [ ]:
# Load metrik dari file evaluasi
summary_path = METRICS_DIR / "evaluation_summary.txt"
if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("evaluation_summary.txt tidak ditemukan. Jalankan 07_evaluasi.ipynb terlebih dahulu.")

In [ ]:
# Hitung ulang metrik untuk tabel perbandingan
from evaluate import classify_indicators, indicator_summary
from sklearn.metrics import f1_score, accuracy_score

y_suit_pred = model.predict_suit(X_test)
y_cat_pred  = model.predict_category(X_test)

indicators = classify_indicators(y_suit_test, y_cat_test, y_suit_pred, y_cat_pred)
summary    = indicator_summary(indicators)

suit_f1  = f1_score(y_suit_test, y_suit_pred, average="weighted", zero_division=0)
cat_f1   = f1_score(y_cat_test,  y_cat_pred,  average="weighted", zero_division=0)
suit_acc = accuracy_score(y_suit_test, y_suit_pred)
cat_acc  = accuracy_score(y_cat_test,  y_cat_pred)

comparison_table = pd.DataFrame({
    "Metrik":     ["SC (Same Category)", "SS (Same Suit)", "MS (Most Suitable)",
                   "Stage-1 Suit F1",    "Stage-2 Category F1",
                   "Stage-1 Suit Acc",   "Stage-2 Category Acc"],
    "C23 Paper":  [0.773, None, None, None, None, None, None],
    "Model Kita": [
        round(summary["SC (utama)"], 4),
        round(summary["SS"],  4),
        round(summary["MS"],  4),
        round(suit_f1,  4),
        round(cat_f1,   4),
        round(suit_acc, 4),
        round(cat_acc,  4),
    ],
})
print("=== Perbandingan dengan C23 Paper ===")
comparison_table

## 8.2 Analisis Kesalahan Prediksi

In [ ]:
pred_test = model.predict(X_test)
error_df = pd.DataFrame({
    "suit_true":  y_suit_test.values,
    "suit_pred":  pred_test["pred_suit"].values,
    "cat_true":   y_cat_test.values,
    "cat_pred":   pred_test["pred_category"].values,
})
error_df["suit_wrong"] = error_df["suit_true"] != error_df["suit_pred"]
error_df["cat_wrong"]  = error_df["cat_true"]  != error_df["cat_pred"]
error_df["both_wrong"] = error_df["suit_wrong"] & error_df["cat_wrong"]

print(f"Board salah prediksi suit     : {error_df['suit_wrong'].sum()} ({error_df['suit_wrong'].mean():.1%})")
print(f"Board salah prediksi kategori : {error_df['cat_wrong'].sum()} ({error_df['cat_wrong'].mean():.1%})")
print(f"Board salah keduanya          : {error_df['both_wrong'].sum()} ({error_df['both_wrong'].mean():.1%})")
print()
print("Pola kesalahan prediksi suit (benar → salah prediksi):")
errors = error_df[error_df["suit_wrong"]][["suit_true", "suit_pred"]]
print(errors.groupby(["suit_true", "suit_pred"]).size().sort_values(ascending=False).head(10))

In [ ]:
print("Pola kesalahan prediksi kategori (benar → salah prediksi):")
errors_cat = error_df[error_df["cat_wrong"]][["cat_true", "cat_pred"]]
print(errors_cat.groupby(["cat_true", "cat_pred"]).size().sort_values(ascending=False).head(10))

## 8.3 Kesalahan vs Total HCP

Apakah kesalahan prediksi lebih banyak pada HCP borderline (sekitar 24-26)?

In [ ]:
error_df_idx = error_df.copy()
error_df_idx.index = y_suit_test.index

if "total_hcp" in X_test.columns:
    error_df_idx["total_hcp"] = X_test["total_hcp"].values

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    correct = error_df_idx[~error_df_idx["suit_wrong"]]["total_hcp"]
    wrong   = error_df_idx[ error_df_idx["suit_wrong"]]["total_hcp"]
    axes[0].hist(correct, bins=20, alpha=0.6, color="green", label="Prediksi Benar", density=True)
    axes[0].hist(wrong,   bins=20, alpha=0.6, color="red",   label="Prediksi Salah", density=True)
    axes[0].set_title("Distribusi Total HCP: Benar vs Salah (Stage 1 Suit)")
    axes[0].set_xlabel("Total HCP NS")
    axes[0].legend()

    correct_c = error_df_idx[~error_df_idx["cat_wrong"]]["total_hcp"]
    wrong_c   = error_df_idx[ error_df_idx["cat_wrong"]]["total_hcp"]
    axes[1].hist(correct_c, bins=20, alpha=0.6, color="green", label="Prediksi Benar", density=True)
    axes[1].hist(wrong_c,   bins=20, alpha=0.6, color="red",   label="Prediksi Salah", density=True)
    axes[1].set_title("Distribusi Total HCP: Benar vs Salah (Stage 2 Category)")
    axes[1].set_xlabel("Total HCP NS")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "error_analysis_hcp.png", dpi=150)
    plt.show()
    print("Grafik disimpan ke results/figures/error_analysis_hcp.png")
else:
    print("Kolom total_hcp tidak tersedia di X_test")

## 8.4 Analisis Feature Importance

Mengidentifikasi fitur yang paling berpengaruh untuk masing-masing stage.

In [ ]:
from evaluate import plot_feature_importance

imp_suit, imp_cat = model.feature_importance()
plot_feature_importance(imp_suit, imp_cat, top_n=20, model_name="TwoStageRF")

In [ ]:
# Top 20 fitur per stage dalam format tabel
fi_table = pd.DataFrame({
    "Rank":           range(1, 21),
    "Fitur (Suit)": imp_suit.head(20).index,
    "Importance (Suit)": imp_suit.head(20).values.round(4),
    "Fitur (Category)": imp_cat.head(20).index,
    "Importance (Cat)": imp_cat.head(20).values.round(4),
})
fi_table

## 8.5 Porsi Kontribusi Kelompok Fitur

Seberapa besar kontribusi fitur kartu vs fitur bidding history?

In [ ]:
def group_importance(imp_series):
    groups = {
        "HCP (player/partner)": [f for f in imp_series.index if "hcp" in f and f not in ["total_hcp"]],
        "Total HCP": ["total_hcp"],
        "Suit Count (total_num)": [f for f in imp_series.index if f.startswith("total_num_")],
        "Balance": [f for f in imp_series.index if "balance" in f],
        "Stopper": [f for f in imp_series.index if "stop" in f],
        "Vulnerability": ["vulnerability"],
        "Dealer": [f for f in imp_series.index if f.startswith("dealer_")],
        "Best Fit": [f for f in imp_series.index if "fit" in f],
        "Bidding History (72-bit)": [f for f in imp_series.index if f.startswith("bid_")],
    }
    result = {}
    for name, cols in groups.items():
        total = imp_series[imp_series.index.isin(cols)].sum()
        result[name] = round(total, 4)
    return pd.Series(result).sort_values(ascending=False)

print("Kontribusi kelompok fitur — Stage 1 (Suit):")
print(group_importance(imp_suit).to_string())
print()
print("Kontribusi kelompok fitur — Stage 2 (Category):")
print(group_importance(imp_cat).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, imp, title in [
    (axes[0], imp_suit, "Stage 1: Suit"),
    (axes[1], imp_cat,  "Stage 2: Category"),
]:
    grp = group_importance(imp)
    colors = plt.cm.Set3(range(len(grp)))
    ax.barh(grp.index, grp.values, color=colors)
    ax.set_title(f"Kontribusi Kelompok Fitur — {title}")
    ax.set_xlabel("Total Feature Importance")
    for i, v in enumerate(grp.values):
        ax.text(v + 0.001, i, f"{v:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_group_importance.png", dpi=150)
plt.show()
print("Grafik disimpan ke results/figures/feature_group_importance.png")

## 8.6 Ekspor Hasil Analisis

Simpan semua tabel ke CSV untuk kemudahan copy-paste ke skripsi.

In [ ]:
# Tabel perbandingan dengan C23
comparison_table.to_csv(METRICS_DIR / "comparison_c23.csv", index=False)
print("Disimpan: comparison_c23.csv")

# Tabel feature importance
fi_table.to_csv(METRICS_DIR / "feature_importance_table.csv", index=False)
print("Disimpan: feature_importance_table.csv")

# Tabel pola kesalahan
error_suit_pattern = errors.groupby(["suit_true", "suit_pred"]).size().reset_index(name="count")
error_suit_pattern.to_csv(METRICS_DIR / "error_suit_pattern.csv", index=False)
print("Disimpan: error_suit_pattern.csv")

print(f"\nSemua file analisis tersimpan di: {METRICS_DIR}")

---
## Catatan untuk Penulisan Skripsi

### Bab 3 — Metodologi
- Gambarkan flowchart pipeline dari Step 1 sampai Step 7
- Tabel fitur C23 (ada di `CLAUDE.md` Section "Fitur yang Akan Diekstrak")
- Jelaskan arsitektur 2-stage RF dan alasan pemilihan metode

### Bab 4 — Hasil dan Pembahasan
- Tampilkan tabel 7 indikator (dari `evaluation_summary.txt`)
- Confusion matrix kedua stage (dari `results/figures/`)
- Feature importance plot + analisis grup fitur
- Bandingkan SC kita vs C23 paper (0.773)
- Analisis kesalahan: pola misprediksi + hubungan dengan HCP

### File yang dihasilkan seluruh pipeline:
```
data/parsed/parsed_boards.csv           ← Hasil parsing (Step 1)
data/processed/features.csv             ← Fitur terekstrak (Step 3)
data/processed/labeled_boards.csv       ← Label DDS (Step 4)
data/processed/bridge_dataset.csv       ← Dataset final (Step 5)
results/metrics/dataset_split.pkl       ← Train/test split (Step 5)
results/metrics/rf_model.pkl            ← Model tersimpan (Step 6)
results/metrics/evaluation_summary.txt  ← Ringkasan metrik (Step 7)
results/metrics/comparison_c23.csv      ← Tabel perbandingan (Step 8)
results/metrics/feature_importance_table.csv
results/metrics/error_suit_pattern.csv
results/figures/eda_*.png               ← Grafik EDA (Step 2)
results/figures/label_distribution.png  ← Distribusi label (Step 4)
results/figures/confusion_matrix_*.png  ← Confusion matrices (Step 7)
results/figures/feature_importance_*.png
results/figures/indicators_TwoStageRF.png
results/figures/error_analysis_hcp.png  ← Analisis kesalahan (Step 8)
results/figures/feature_group_importance.png
```